In [1]:
import base64
import io
from datetime import datetime
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine
import dash
from dash import dcc, html, Input, Output, State, callback_context
import dash_bootstrap_components as dbc
from expectation import Expectation, DataFrameContext  # Votre module de vérification

# Création de l'application Dash avec le thème Bootstrap
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
server = app.server

# Variable globale pour stocker le DataFrame chargé
global_df = None

# --- Layout du Sidebar (menu latéral) ---
sidebar = html.Div(
    [
        html.Img(src="https://upload.wikimedia.org/wikipedia/commons/6/6b/Pandas_logo.svg", style={"width": "100px"}),
        html.Hr(),
        dbc.Nav(
            [
                dbc.NavLink("🏠 Accueil", href="/", active="exact"),
                dbc.NavLink("📊 Analyse Qualité", href="/analyse-qualite", active="exact"),
                dbc.NavLink("⚙️ Paramètres", href="/parametres", active="exact"),
                dbc.NavLink("Rapport Qualité", href="/rapport-qualite", active="exact"),
            ],
            vertical=True,
            pills=True,
        ),
        html.Hr(),
        html.Div(
            [
                html.Label("Source des données :"),
                dcc.RadioItems(
                    id="data-source",
                    options=[
                        {"label": "Fichier CSV/Excel", "value": "file"},
                        {"label": "Base SQL", "value": "sql"}
                    ],
                    value="file",
                    labelStyle={"display": "block", "margin": "5px 0"}
                ),
                html.Div(id="data-source-inputs"),
            ],
            style={"padding": "10px"}
        ),
    ],
    style={
        "position": "fixed", "top": 0, "left": 0, "bottom": 0, "width": "250px",
        "padding": "20px", "background-color": "#f8f9fa"
    },
)

# --- Layout de la zone de contenu principal ---
content = html.Div(id="page-content", style={"margin-left": "270px", "margin-right": "20px", "padding": "20px"})

# Layout global de l'app
app.layout = html.Div([dcc.Location(id="url"), sidebar, content])

# --- Callback pour afficher les entrées selon la source de données ---
@app.callback(
    Output("data-source-inputs", "children"),
    Input("data-source", "value"),
)
def update_data_source_inputs(value):
    if value == "file":
        return dcc.Upload(
            id="upload-data",
            children=html.Div(["Drag and Drop ou ", html.A("Sélectionnez un fichier")]),
            style={
                "width": "100%", "height": "60px", "lineHeight": "60px",
                "borderWidth": "1px", "borderStyle": "dashed", "borderRadius": "5px",
                "textAlign": "center"
            },
            multiple=False,
        )
    elif value == "sql":
        return html.Div([
            dbc.Input(id="db-url", placeholder="URL de la base de données", type="text", value="sqlite:///example.db", style={"margin-bottom": "10px"}),
            dbc.Textarea(id="sql-query", placeholder="Requête SQL", value="SELECT * FROM clients", style={"margin-bottom": "10px"}),
            dbc.Button("Charger les données", id="load-sql", color="primary")
        ])

# --- Callback pour charger les données (depuis fichier ou SQL) ---
@app.callback(
    Output("page-content", "children"),
    Input("url", "pathname"),
    Input("upload-data", "contents"),
    Input("upload-data", "filename"),
    Input("load-sql", "n_clicks"),
    State("db-url", "value"),
    State("sql-query", "value"),
    prevent_initial_call=True
)
def render_page_content(pathname, contents, filename, sql_clicks, db_url, sql_query):
    ctx = callback_context
    global global_df
    triggered = ctx.triggered[0]["prop_id"].split(".")[0] if ctx.triggered else ""
    if triggered == "upload-data" and contents is not None:
        content_type, content_string = contents.split(',')
        decoded = base64.b64decode(content_string)
        try:
            if filename.lower().endswith("csv"):
                global_df = pd.read_csv(io.StringIO(decoded.decode("utf-8")))
            elif filename.lower().endswith(("xls", "xlsx")):
                global_df = pd.read_excel(io.BytesIO(decoded))
        except Exception as e:
            return html.Div(["Erreur lors du traitement du fichier."])
    elif triggered == "load-sql":
        try:
            engine = create_engine(db_url)
            global_df = pd.read_sql(sql_query, engine)
        except Exception as e:
            return html.Div([f"Erreur lors du chargement SQL: {e}"])
    
    # Affichage selon l'URL
    if pathname in ["/", "/accueil"]:
        if global_df is None:
            return html.Div(["Aucune donnée chargée. Veuillez charger un fichier ou une base SQL."])
        # Création de cartes KPI avec Dash Bootstrap Components
        kpi_cards = dbc.Row([
            dbc.Col(dbc.Card(dbc.CardBody([
                html.H4("Nombre d'enregistrements", className="card-title"),
                html.P(str(global_df.shape[0]), className="card-text")
            ]), color="light"), width=3),
            dbc.Col(dbc.Card(dbc.CardBody([
                html.H4("Taux de validité", className="card-title"),
                html.P("95%", className="card-text")
            ]), color="light"), width=3),
            dbc.Col(dbc.Card(dbc.CardBody([
                html.H4("Taux d'erreurs", className="card-title"),
                html.P("5%", className="card-text")
            ]), color="light"), width=3),
            dbc.Col(dbc.Card(dbc.CardBody([
                html.H4("Dernière mise à jour", className="card-title"),
                html.P(datetime.today().strftime("%Y-%m-%d"), className="card-text")
            ]), color="light"), width=3),
        ])
        return html.Div([html.H2("Accueil"), kpi_cards])
    
    elif pathname == "/analyse-qualite":
        if global_df is None:
            return html.Div(["Aucune donnée chargée. Veuillez charger un fichier ou une base SQL."])
        tabs = dcc.Tabs(id="tabs-analysis", value="tab-expectation", children=[
            dcc.Tab(label="Vérification par Expectation", value="tab-expectation"),
            dcc.Tab(label="Analyse Complémentaire", value="tab-complement"),
        ])
        return html.Div([
            html.H2("Analyse de la Qualité des Données"),
            tabs,
            html.Div(id="tabs-content")
        ])
    
    elif pathname == "/parametres":
        return html.Div([
            html.H2("Paramètres du Dashboard"),
            dbc.Button("Recharger les données", id="reload-data", color="primary"),
            html.Br(), html.Br(),
            dcc.RadioItems(
                id="theme-selector",
                options=[{"label": "Clair", "value": "light"}, {"label": "Sombre", "value": "dark"}],
                value="light",
                labelStyle={"display": "inline-block", "margin-right": "10px"}
            ),
            html.Div(id="theme-output")
        ])
    
    elif pathname == "/rapport-qualite":
        if global_df is None:
            return html.Div(["Aucune donnée chargée. Veuillez charger un fichier ou une base SQL."])
        return html.Div([
            html.H2("Rapport Qualité"),
            dcc.Tabs(id="rapport-tabs", value="rapport-config", children=[
                dcc.Tab(label="Configuration des Vérifications", value="rapport-config"),
                dcc.Tab(label="Résultats des Vérifications", value="rapport-results"),
            ]),
            html.Div(id="rapport-tabs-content")
        ])
    else:
        return html.Div([html.H2("404: Not found"), html.P("La page demandée n'existe pas.")])

# --- Callback pour le contenu des onglets de la page Analyse Qualité ---
@app.callback(
    Output("tabs-content", "children"),
    Input("tabs-analysis", "value")
)
def render_analysis_tab(tab_value):
    if tab_value == "tab-expectation":
        return html.Div([
            html.H3("Vérification des règles de qualité avec Expectation"),
            dcc.Dropdown(
                id="expectation-function",
                options=[{"label": f, "value": f} for f in [
                    "ExpectColumnValuesToNotBeNull",
                    "ExpectColumnValuesToBeBetween",
                    "ExpectColumnValuesToMatchRegex",
                    "ExpectColumnValuesToBeUnique",
                    "ExpectColumnValuesToBeWithinIQR",
                    "ExpectColumnValuesToBeInSet",
                    "ExpectColumnValuesToHaveLengthBetween",
                    "ExpectColumnValuesToRespectOrder",
                    "ExpectColumnValuesToBeInDateRange",
                    "ExpectRowCompletenessToBeAboveThreshold"
                ]],
                placeholder="Choisissez une fonction de vérification"
            ),
            html.Div(id="expectation-params", style={"margin-top": "10px"}),
            dbc.Button("Lancer la vérification", id="run-expectation", color="primary", style={"margin-top": "10px"}),
            html.Div(id="expectation-output", style={"margin-top": "20px"})
        ])
    elif tab_value == "tab-complement":
        # Exemple d'analyse complémentaire
        total_values = global_df.shape[0] * global_df.shape[1]
        missing_percent = (global_df.isnull().sum().sum() / total_values) * 100
        duplicate_percent = (global_df.shape[0] - global_df.drop_duplicates().shape[0]) / global_df.shape[0] * 100
        metrics = dbc.Row([
            dbc.Col(dbc.Card(dbc.CardBody([html.H5("Valeurs Manquantes"), html.P(f"{missing_percent:.2f}%")])), width=6),
            dbc.Col(dbc.Card(dbc.CardBody([html.H5("Doublons"), html.P(f"{duplicate_percent:.2f}%")])), width=6)
        ])
        data_preview = dbc.Table.from_dataframe(global_df.head(), striped=True, bordered=True, hover=True)
        return html.Div([html.H3("Analyse Complémentaire"), metrics, html.Hr(), html.H5("Aperçu des données"), data_preview])

# --- Callback pour afficher les paramètres de la fonction Expectation sélectionnée ---
@app.callback(
    Output("expectation-params", "children"),
    Input("expectation-function", "value")
)
def render_expectation_params(func):
    if not func:
        return ""
    children = []
    # Pour les fonctions nécessitant la sélection d'une colonne
    if func in ["ExpectColumnValuesToNotBeNull", "ExpectColumnValuesToBeUnique"]:
        children.append(dcc.Dropdown(
            id="expectation-column",
            options=[{"label": col, "value": col} for col in global_df.columns],
            placeholder="Sélectionnez une colonne"
        ))
    elif func == "ExpectColumnValuesToBeBetween":
        children.append(dcc.Dropdown(
            id="expectation-column",
            options=[{"label": col, "value": col} for col in global_df.columns],
            placeholder="Sélectionnez une colonne"
        ))
        children.append(dcc.Input(id="min-value", type="number", placeholder="Valeur minimale", value=0))
        children.append(dcc.Input(id="max-value", type="number", placeholder="Valeur maximale", value=100))
    elif func == "ExpectColumnValuesToMatchRegex":
        children.append(dcc.Dropdown(
            id="expectation-column",
            options=[{"label": col, "value": col} for col in global_df.columns],
            placeholder="Sélectionnez une colonne"
        ))
        children.append(dcc.Input(id="regex", type="text", placeholder="Regex", value=r"^[\w\.-]+@[\w\.-]+\.\w+$"))
    elif func == "ExpectColumnValuesToBeWithinIQR":
        children.append(dcc.Dropdown(
            id="expectation-column",
            options=[{"label": col, "value": col} for col in global_df.columns],
            placeholder="Sélectionnez une colonne"
        ))
        children.append(dcc.Input(id="factor", type="number", placeholder="Facteur", value=1.5))
    elif func == "ExpectColumnValuesToBeInSet":
        children.append(dcc.Dropdown(
            id="expectation-column",
            options=[{"label": col, "value": col} for col in global_df.columns],
            placeholder="Sélectionnez une colonne"
        ))
        children.append(dcc.Input(id="valid-values", type="text", placeholder="Valeurs autorisées séparées par des virgules", value="A,B,C"))
    elif func == "ExpectColumnValuesToHaveLengthBetween":
        children.append(dcc.Dropdown(
            id="expectation-column",
            options=[{"label": col, "value": col} for col in global_df.columns],
            placeholder="Sélectionnez une colonne"
        ))
        children.append(dcc.Input(id="min-length", type="number", placeholder="Longueur minimale", value=1))
        children.append(dcc.Input(id="max-length", type="number", placeholder="Longueur maximale", value=100))
    elif func == "ExpectColumnValuesToRespectOrder":
        children.append(dcc.Dropdown(
            id="expectation-column",
            options=[{"label": col, "value": col} for col in global_df.columns],
            placeholder="Sélectionnez la première colonne"
        ))
        children.append(dcc.Dropdown(
            id="expectation-column2",
            options=[{"label": col, "value": col} for col in global_df.columns],
            placeholder="Sélectionnez la seconde colonne"
        ))
    elif func == "ExpectColumnValuesToBeInDateRange":
        children.append(dcc.Dropdown(
            id="expectation-column",
            options=[{"label": col, "value": col} for col in global_df.columns],
            placeholder="Sélectionnez une colonne"
        ))
        children.append(dcc.DatePickerSingle(id="min-date", date="2000-01-01"))
        children.append(dcc.DatePickerSingle(id="max-date", date=datetime.today().strftime("%Y-%m-%d")))
    elif func == "ExpectRowCompletenessToBeAboveThreshold":
        children.append(dcc.Input(id="threshold", type="number", placeholder="Seuil (0 à 1)", value=0.8))
    return html.Div(children, style={"margin-top": "10px"})

# --- Callback pour lancer la vérification via Expectation ---
@app.callback(
    Output("expectation-output", "children"),
    Input("run-expectation", "n_clicks"),
    State("expectation-function", "value"),
    State("expectation-column", "value"),
    State("min-value", "value"),
    State("max-value", "value"),
    State("regex", "value"),
    State("factor", "value"),
    State("valid-values", "value"),
    State("min-length", "value"),
    State("max-length", "value"),
    State("expectation-column2", "value"),
    State("min-date", "date"),
    State("max-date", "date"),
    State("threshold", "value"),
    prevent_initial_call=True
)
def run_expectation(n_clicks, func, column, min_value, max_value, regex, factor, valid_values, min_length, max_length, column2, min_date, max_date, threshold):
    if global_df is None or not func:
        return "Aucune donnée ou fonction sélectionnée."
    params = {}
    if func in ["ExpectColumnValuesToNotBeNull", "ExpectColumnValuesToBeUnique"]:
        params["column"] = column
    elif func == "ExpectColumnValuesToBeBetween":
        params["column"] = column
        params["min_value"] = min_value
        params["max_value"] = max_value
    elif func == "ExpectColumnValuesToMatchRegex":
        params["column"] = column
        params["regex"] = regex
    elif func == "ExpectColumnValuesToBeWithinIQR":
        params["column"] = column
        params["factor"] = factor
    elif func == "ExpectColumnValuesToBeInSet":
        params["column"] = column
        params["valid_values"] = [v.strip() for v in valid_values.split(",") if v.strip()]
    elif func == "ExpectColumnValuesToHaveLengthBetween":
        params["column"] = column
        params["min_length"] = int(min_length)
        params["max_length"] = int(max_length)
    elif func == "ExpectColumnValuesToRespectOrder":
        params["column1"] = column
        params["column2"] = column2
    elif func == "ExpectColumnValuesToBeInDateRange":
        params["column"] = column
        params["min_date"] = pd.to_datetime(min_date)
        params["max_date"] = pd.to_datetime(max_date)
    elif func == "ExpectRowCompletenessToBeAboveThreshold":
        params["threshold"] = threshold

    with DataFrameContext(global_df) as ctx:
        exp_func = getattr(Expectation, func)
        try:
            ctx.apply_expectation(exp_func, **params)
        except Exception as e:
            return f"Erreur: {e}"
    results = ctx.reports
    results_df = pd.DataFrame(results)
    results_df["unexpected_list"] = results_df["unexpected_list"].apply(
        lambda x: ", ".join(map(str, x)) if isinstance(x, list) else str(x)
    )
    table = dbc.Table.from_dataframe(results_df, striped=True, bordered=True, hover=True)
    if not results_df.empty:
        fig_bar = px.bar(results_df, x="check", y="unexpected_count", 
                         title="Nombre d'erreurs détectées par règle", 
                         color="unexpected_count", color_continuous_scale="reds")
    else:
        fig_bar = {}
    return html.Div([
        html.H4("Rapport de validation"),
        table,
        dcc.Graph(figure=fig_bar)
    ])

# --- Callback pour les onglets du Rapport Qualité (configuration et résultats) ---
@app.callback(
    Output("rapport-tabs-content", "children"),
    Input("rapport-tabs", "value")
)
def render_rapport_tab(tab_value):
    if global_df is None:
        return "Aucune donnée chargée."
    if tab_value == "rapport-config":
        return html.Div([
            html.H4("Configuration des Vérifications"),
            html.P("Ici, vous pouvez configurer les vérifications par colonne (interface à adapter selon vos besoins).")
        ])
    elif tab_value == "rapport-results":
        return html.Div([
            html.H4("Résultats des Vérifications"),
            html.P("Les résultats seront affichés ici après validation.")
        ])

if __name__ == '__main__':
    app.run_server(debug=True)
